# GRAG downstream: ExplaGraphs classification on Colab

**Input:** a completed, checksummed Kaggle compact preprocessing backup in Google Drive.
No sentence encoder, ego-graph retrieval, or preprocessing needs to run again.

**Model:** `meta-llama/Llama-2-7b-chat-hf` with NF4 4-bit QLoRA.
Train LoRA adapters **and** the repository's GAT graph encoder/projector.
The quantized Llama backbone stays frozen.

**Output:** resumable checkpoints, raw predictions, metrics and experiment settings,
automatically copied to Drive. The notebook starts with a small smoke run.

This is a T4-conscious adaptation, **not an exact paper reproduction**.
Chat weights, quantization, prompt limits, training duration and preprocessing
choices must be recorded when comparing results.


## 1. Before starting

1. Select **Runtime > Change runtime type > T4 GPU**.
2. Accept the Llama-2 model license on Hugging Face using your own account.
3. Add a read-access token named **HF_TOKEN** in Colab Secrets.
   Enable notebook access. Never paste the token into a cell.
4. Ensure the updated repository, including `src/utils/notebook_downstream/`,
   has been pushed or uploaded before cloning it here.
5. Your preprocessing snapshot must contain **manifest.json** and every archive part.

We reuse Colab's installed PyTorch/CUDA. No isolated environment, `torch_scatter`,
`wandb`, or author-era package stack is required.


In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime, then rerun this cell.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}; memory: {gpu.total_memory / 2**30:.2f} GiB")
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)

REPO_URL = "https://github.com/MichaelPham2005/BioGRAG-QA.git"
REPO_BRANCH = "GRAG/reproduce/WebQSP"
ROOT = Path("/content/BioGRAG-QA")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH,
                    "--single-branch", REPO_URL, str(ROOT)], check=True)
if not (ROOT / "src/utils/notebook_downstream/training.py").is_file():
    raise FileNotFoundError("Updated downstream helpers are missing. Push/upload them first.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


## 2. Install only missing dependencies

Do not reinstall PyTorch or CUDA.
If a package was already imported before installation and Colab requests a restart,
restart the session and rerun from section 1.


In [ ]:
requirements = {
    "torch_geometric": "torch-geometric",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "peft": "peft",
    "sentencepiece": "sentencepiece",
    "pandas": "pandas",
}
missing = [package for module, package in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
from importlib.metadata import version
print({name: version(name) for name in requirements.values()})

from src.utils.notebook_downstream import artifacts, training


## 3. Mount Drive and select a completed expla_graphs snapshot

The default folder is the one created by the Kaggle backup helper.
If you have several snapshots, copy the desired folder name into `SNAPSHOT_NAME`.
A missing manifest indicates an incomplete upload; do not restore that folder.

Archive and file sizes/checksums are verified.
Links, unexpected files and path traversal are rejected.
Restored paths are local Colab paths; producer hashes and configuration are
preserved without rewriting the old Kaggle paths.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DATASET = "expla_graphs"
BACKUP_ROOT = Path("/content/drive/MyDrive/GRAG preprocessing backups")
SNAPSHOT_NAME = ""  # Set this explicitly when more than one matching snapshot exists.
candidates = []
for path in sorted(BACKUP_ROOT.glob("*/manifest.json")):
    if json.loads(path.read_text()).get("dataset") == DATASET:
        candidates.append(path.parent)
print("Completed snapshots:", [path.name for path in candidates])
if SNAPSHOT_NAME:
    selected = [path for path in candidates if path.name == SNAPSHOT_NAME]
else:
    selected = candidates
if len(selected) != 1:
    raise ValueError("Set SNAPSHOT_NAME to exactly one of the listed folders, then rerun.")
SNAPSHOT = selected[0]
PREPARED, PREPROCESS_RUN, SNAPSHOT_HASH = artifacts.restore_snapshot(
    SNAPSHOT, ROOT / "restored" / DATASET / SNAPSHOT.name, DATASET,
)
dataset = artifacts.CompactTrainingDataset(PREPARED, PREPROCESS_RUN)
original_splits = dataset.get_idx_split()
print("Retained split counts:", {key: len(ids) for key, ids in original_splits.items()})
print("Excluded source rows:", dataset.metadata.get("excluded_count", 0))
print("Prepared:", PREPARED)
print("Compact samples:", PREPROCESS_RUN)


## 4. Configure an experiment

- Leave `SMOKE_TEST=True` for the first run: up to 8 train / 4 validation / 4 test examples.
- After that works, set **SMOKE_TEST=False** and rerun from this section.
  The default run name changes, so smoke checkpoints cannot contaminate the full run.
- Full mode uses the original retained split membership and **3 epochs** by default,
  not the repository's 10-epoch setting.
- Microbatch 1 and accumulation 4 give an effective batch of up to 4.
  The final short group is normalized by its actual microbatch count.
- Use a new `RUN_NAME` after changing settings, input snapshot, or helper code.

Drive backup is always enabled here. An interrupted runtime can resume at the last
backed-up optimizer-step boundary (default every 50 updates), or completed epoch.
Keep the same snapshot, run name and settings when resuming.


In [ ]:
SMOKE_TEST = True
RUN_NAME = "expla-graphs-chat-qlora-" + ("smoke-v2" if SMOKE_TEST else "full-v2")
OUTPUT = ROOT / "output" / "downstream" / DATASET / RUN_NAME
DRIVE_OUTPUT = Path("/content/drive/MyDrive/GRAG downstream") / DATASET / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)
if not (OUTPUT / "run.json").exists():
    artifacts.restore_training_outputs(DRIVE_OUTPUT, OUTPUT)

limits = {"train": 8, "val": 4, "test": 4}
splits = {name: list(ids[:limits[name]] if SMOKE_TEST else ids)
          for name, ids in original_splits.items()}
saved_run = json.loads((OUTPUT / "run.json").read_text()) if (OUTPUT / "run.json").exists() else None
print("Selected split counts:", {key: len(ids) for key, ids in splits.items()})
print("Drive output folder:", DRIVE_OUTPUT)


## 5. Pin and load the chat model

**Why chat?** It is instruction-tuned and is explicitly named in the paper's
no-retrieval baseline comparison. This alone does not establish that every
experiment used that checkpoint. The repository's default is the base model;
this notebook deliberately selects the chat model.

**What learns?**

- LoRA rank **8**, alpha **16**, dropout **0.05**, targeting `q_proj` and `v_proj`.
- GAT: **4 layers, 4 heads, hidden width 1024** with query-conditioned scaling.
- Projector: **1024 -> 2048 -> Llama hidden width**, producing one soft graph token.

QLoRA adds adapters to frozen NF4 weights; it does not fully fine-tune 7 billion
parameters. Non-reentrant checkpointing reduces activation memory while keeping
the gradient path through Llama to the graph prompt.

The instruction delimiters follow the repository's `[INST] ... [/INST]` layout.
The graph token precedes the description and question inside that instruction.
Only answer/EOS tokens receive training labels; test generation receives no gold answer.


In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi
import random

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Add HF_TOKEN to Colab Secrets and enable notebook access.")
MODEL_ID = "meta-llama/Llama-2-7b-chat-hf"
MODEL_REVISION = ""  # Optional commit SHA; otherwise pin main once, or reuse the saved SHA.
requested_revision = MODEL_REVISION or (
    saved_run["settings"]["model_revision"] if saved_run else "main"
)
resolved_revision = HfApi(token=HF_TOKEN).model_info(
    MODEL_ID, revision=requested_revision,
).sha
settings = training.Settings(
    dataset=DATASET, model_id=MODEL_ID, model_revision=resolved_revision,
    epochs=1 if SMOKE_TEST else 3, batch_size=1, grad_accum=4,
    max_sequence_tokens=512, max_description_tokens=256,
    max_answer_tokens=8,
)
SIGNATURE = training.initialize_run(
    settings, dataset, splits, SNAPSHOT_HASH, OUTPUT, DRIVE_OUTPUT,
)
random.seed(settings.seed)
torch.manual_seed(settings.seed)
torch.cuda.manual_seed_all(settings.seed)
model = training.load_qlora(settings, HF_TOKEN)
llm_trainable, llm_total = model.llm.get_nb_trainable_parameters()
graph_trainable = sum(p.numel() for p in model.graph_encoder.parameters() if p.requires_grad)
projector_trainable = sum(p.numel() for p in model.projector.parameters() if p.requires_grad)
print("Pinned model revision:", resolved_revision)
print({"LLM logical parameters (includes adapters)": llm_total,
       "trainable LoRA": llm_trainable, "trainable GAT": graph_trainable,
       "trainable projector": projector_trainable})
print("HF_TOKEN is not included in the saved experiment/checkpoints.")


## 6. Check text limits before training

Context budget: **512 tokens**, including the graph token and answer reserve.
Descriptions are capped at 256 tokens; questions are never silently truncated.
Over-budget questions raise an error. Answer targets are capped and reported below.
Gold labels used for scoring remain untruncated.

This preflight reads compact payloads one at a time without expanding graph features.
Large graph topology is **not** truncated to meet the text budget.


In [ ]:
report = training.token_report(model, dataset, splits["train"] + splits["val"])
print("Training + validation token-limit report:", report)
training.save_and_backup(report, "token-report.json", OUTPUT, DRIVE_OUTPUT)


## 7. Train, validate and back up

The best checkpoint is selected by **validation loss**, never test performance.
Early stopping uses validation loss with patience 3.
Correct accumulation, FP16 loss scaling, finite-gradient checks and clipping are enabled.
AMP starts at loss scale 1024. On gradient overflow it lowers the scale and retries
the same accumulation group, restoring BatchNorm statistics and RNG state.
Invalid gradients never update weights or advance the training cursor.
Repeated failure stops after 16 attempts and identifies the affected parameters.
The v2 run names keep this fix separate from older helper fingerprints/checkpoints.

`last.pt` contains adapters, graph modules, optimizer/scaler, RNG and progress.
`best.pt` contains the best adapters and graph modules, including BatchNorm buffers.
Neither contains the base Llama weights. Checkpoints are trusted files from your
own run, loaded in PyTorch's restricted `weights_only` mode.

A single-node training microbatch uses BatchNorm's running statistics with a warning.
The original GAT implementation is reused rather than silently replaced.
Its `GATConv` layers do not configure `edge_dim`; in current PyG, relation features
are not directly included in attention. The text description still contains relations.
This source behavior is another reason not to call this an exact paper reproduction.

**Memory:** fitting every WebQSP retrieved graph on a T4 is not guaranteed.
First reduce text limits if needed. If graph activations are the bottleneck,
changing GNN width changes the experiment and requires a new run name.
Do not silently drop large graphs or treat a smoke run as a full result.


In [ ]:
import time

torch.cuda.reset_peak_memory_stats()
started = time.monotonic()
history = training.fit(model, dataset, splits, SIGNATURE, OUTPUT, DRIVE_OUTPUT)
print(f"Training/resume elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
print(f"Peak allocated GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
print("Best validation checkpoint loaded. History:", history)


## 8. Evaluate the held-out test split

Best validation weights are reloaded before testing.
Raw predictions are backed up every 10 examples and at completion.
A prediction cache is tied to the exact best-checkpoint hash and selected test IDs.

ExplaGraphs reports accuracy and macro-F1 as fractions, plus invalid outputs. The first standalone support/counter word is parsed case-insensitively. Invalid outputs count as wrong. This is stricter than the source substring parser.

The retrieved explanation-graph preprocessing is intentionally different from the authors' original ExplaGraphs baseline.


In [ ]:
predictions, metrics = training.predict_test(
    model, dataset, splits["test"], SIGNATURE, OUTPUT, DRIVE_OUTPUT,
)
print(json.dumps(metrics, indent=2))
print("Outputs are saved locally and backed up to:", DRIVE_OUTPUT)


## 9. Inspect predictions and reuse the trained model

Each prediction retains the input question, full gold label, raw generated text,
processing ID and available source identity.
Older ExplaGraphs snapshots may lack native source IDs; these remain `None`.

The trained object `model` is ready for inference on **preprocessed graphs**.
It is not a text-only classification/QA endpoint: a new question also needs its
own retrieved graph and embeddings from a compatible preprocessing pipeline.

To reload in a fresh Colab session, rerun sections 1-5 with the same configuration,
then call `training.load_best(model, OUTPUT, SIGNATURE)` and `model.eval()`.
The gated chat backbone is downloaded/cached again; adapters and graph modules
come from Drive. Checkpoints are not standalone Llama models.


In [ ]:
for row in predictions[:3]:
    print("Processing ID:", row["id"], "Source ID:", row["source_id"])
    print(row["question"])
    print("Gold:", row["label"])
    print("Prediction:", row["prediction"])
    print("-" * 60)
